In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

df = pd.read_csv("/content/sample_data/airline-passengers.csv")
series = df.iloc[:, 1].values.astype("float32").reshape(-1, 1)

print("Dataset shape:", series.shape)
print(df.head())

plt.figure(figsize=(10, 4))
plt.plot(series)
plt.title("International Airline Passengers (1949-1960)")
plt.xlabel("Month")
plt.ylabel("Passengers (thousands)")
plt.show()

scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(series)

LOOK_BACK = 12

def create_dataset(data, look_back=1):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i + look_back, 0])
        y.append(data[i + look_back, 0])
    return np.array(X), np.array(y)

X, y = create_dataset(scaled, LOOK_BACK)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train = X_train.reshape((X_train.shape[0], LOOK_BACK, 1))
X_test = X_test.reshape((X_test.shape[0], LOOK_BACK, 1))

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

def train_and_evaluate(model, name, epochs=100):
    model.compile(optimizer="adam", loss="mse")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=8,
        verbose=0
    )

    train_pred = model.predict(X_train, verbose=0)
    test_pred = model.predict(X_test, verbose=0)

    train_pred_inv = scaler.inverse_transform(train_pred)
    y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))
    test_pred_inv = scaler.inverse_transform(test_pred)
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

    train_rmse = np.sqrt(mean_squared_error(y_train_inv, train_pred_inv))
    test_rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred_inv))

    print(f"\n{name} - Train RMSE: {train_rmse:.2f} | Test RMSE: {test_rmse:.2f}")

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{name} - Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()

    return test_pred_inv, y_test_inv, test_rmse

rnn_model = Sequential([
    SimpleRNN(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
rnn_pred, y_test_inv, rnn_rmse = train_and_evaluate(rnn_model, "SimpleRNN")

lstm_model = Sequential([
    LSTM(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
lstm_pred, _, lstm_rmse = train_and_evaluate(lstm_model, "LSTM")

gru_model = Sequential([
    GRU(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
gru_pred, _, gru_rmse = train_and_evaluate(gru_model, "GRU")

plt.figure(figsize=(10, 5))
plt.plot(y_test_inv, label="Actual", linewidth=2)
plt.plot(rnn_pred, label=f"SimpleRNN (RMSE={rnn_rmse:.1f})")
plt.plot(lstm_pred, label=f"LSTM (RMSE={lstm_rmse:.1f})")
plt.plot(gru_pred, label=f"GRU (RMSE={gru_rmse:.1f})")
plt.title("Airline Passengers Forecast: RNN vs LSTM vs GRU")
plt.xlabel("Time Step (Test Set)")
plt.ylabel("Passengers (thousands)")
plt.legend()
plt.show()

print(f"\nSimpleRNN RMSE: {rnn_rmse:.2f}")
print(f"LSTM RMSE: {lstm_rmse:.2f}")
print(f"GRU RMSE: {gru_rmse:.2f}")

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Embedding, LSTM, Add, Dropout

SEED = 42
np.random.seed(SEED)

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train.astype("float32")[..., np.newaxis] / 255.0
x_test = x_test.astype("float32")[..., np.newaxis] / 255.0

class_names = ["t-shirt", "trouser", "pullover", "dress", "coat","sandal", "shirt", "sneaker", "bag", "ankle boot"]

def make_caption(label_idx):
    name = class_names[label_idx]
    article = "an" if name[0] in "aeiou" else "a"
    return f"startseq a photo of {article} {name} endseq"

N_TRAIN, N_TEST = 4000, 500
train_idx = np.random.choice(len(x_train), N_TRAIN, replace=False)
test_idx = np.random.choice(len(x_test), N_TEST, replace=False)

train_images = x_train[train_idx]
train_captions = [make_caption(int(y_train[i])) for i in train_idx]
test_images = x_test[test_idx]
test_captions = [make_caption(int(y_test[i])) for i in test_idx]

print("Example captions:", train_captions[:3])

tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(train_captions)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(c.split()) for c in train_captions)

print(f"Vocab size: {vocab_size}, Max caption length: {max_len}")

def encode_captions(captions):
    return tokenizer.texts_to_sequences(captions)

train_seqs = encode_captions(train_captions)
test_seqs = encode_captions(test_captions)

def build_sequences(images, seqs):
    X_img, X_seq, y = [], [], []
    for img, seq in zip(images, seqs):
        for t in range(1, len(seq)):
            in_seq, out_word = seq[:t], seq[t]
            in_seq = pad_sequences([in_seq], maxlen=max_len)[0]
            X_img.append(img)
            X_seq.append(in_seq)
            y.append(out_word)
    return np.array(X_img), np.array(X_seq), to_categorical(y, num_classes=vocab_size)

X_img_train, X_seq_train, y_train_cap = build_sequences(train_images, train_seqs)

print("Training pairs:", X_img_train.shape[0])

EMBED_DIM = 128

image_input = Input(shape=(28, 28, 1), name="image_input")
cnn = Conv2D(32, (3, 3), activation="relu", padding="same")(image_input)
cnn = MaxPooling2D()(cnn)
cnn = Conv2D(64, (3, 3), activation="relu", padding="same")(cnn)
cnn = MaxPooling2D()(cnn)
cnn = GlobalAveragePooling2D()(cnn)
image_features = Dense(EMBED_DIM, activation="relu")(cnn)
image_features = Dropout(0.3)(image_features)

seq_input = Input(shape=(max_len,), name="caption_input")
seq_embed = Embedding(vocab_size, EMBED_DIM, mask_zero=True)(seq_input)
seq_embed = Dropout(0.3)(seq_embed)
seq_features = LSTM(EMBED_DIM)(seq_embed)

merged = Add()([image_features, seq_features])
merged = Dense(EMBED_DIM, activation="relu")(merged)
output = Dense(vocab_size, activation="softmax")(merged)

caption_model = Model(inputs=[image_input, seq_input], outputs=output)
caption_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

caption_model.summary()

history = caption_model.fit(
    [X_img_train, X_seq_train],
    y_train_cap,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Caption Model Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical Cross-Entropy")
plt.legend()
plt.show()

index_to_word = {idx: word for word, idx in tokenizer.word_index.items()}

def generate_caption(image, max_length=max_len):
    in_text = "startseq"
    for _ in range(max_length):
        seq = tokenizer.texts_to_sequences([in_text])[0]
        seq = pad_sequences([seq], maxlen=max_length)
        pred = caption_model.predict([np.expand_dims(image, 0), seq], verbose=0)
        next_idx = int(np.argmax(pred))
        next_word = index_to_word.get(next_idx, "")
        if next_word == "" or next_word == "endseq":
            break
        in_text += " " + next_word
    return in_text.replace("startseq", "").strip()

sample_indices = np.random.choice(len(test_images), 6, replace=False)

for idx in sample_indices:
    img = test_images[idx]
    true_label = class_names[int(y_test[test_idx[idx]])]
    pred_caption = generate_caption(img)

    plt.figure(figsize=(4, 4))
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(f"True: {true_label}\nGenerated: {pred_caption}")
    plt.axis("off")
    plt.show()

print("Example generated caption:", generate_caption(test_images[0]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

x_train = pad_sequences(x_train, maxlen=max_len, padding="post")
x_test = pad_sequences(x_test, maxlen=max_len, padding="post")

x_train = x_train.astype("float32") / vocab_size
x_test = x_test.astype("float32") / vocab_size

input_layer = Input(shape=(max_len,))
encoded = Dense(128, activation="relu")(input_layer)
encoded = Dense(64, activation="relu")(encoded)
latent = Dense(32, activation="relu", name="latent_layer")(encoded)

decoded = Dense(64, activation="relu")(latent)
decoded = Dense(128, activation="relu")(decoded)
decoded = Dense(max_len, activation="sigmoid")(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.fit(
    x_train, x_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

encoder = Model(input_layer, latent)

X_train_features = encoder.predict(x_train, verbose=0)
X_test_features = encoder.predict(x_test, verbose=0)

classifier_input = Input(shape=(32,))
x = Dense(16, activation="relu")(classifier_input)
output = Dense(1, activation="sigmoid")(x)

classifier = Model(classifier_input, output)
classifier.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

classifier.fit(
    X_train_features, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

y_prob = classifier.predict(X_test_features, verbose=0)
y_pred = (y_prob > 0.5).astype(int).ravel()

print("\nSentiment Classification Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

original_dim = 784
latent_dim = 2
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

class VAE(nn.Module):
    def __init__(self, original_dim=784, latent_dim=2):
        super().__init__()
        self.enc_fc1 = nn.Linear(original_dim, 256)
        self.enc_fc2 = nn.Linear(256, 64)
        self.fc_mean = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, 64)
        self.dec_fc2 = nn.Linear(64, 256)
        self.dec_out = nn.Linear(256, original_dim)

    def encode(self, x):
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.fc_mean(h), self.fc_logvar(h)

    def reparameterize(self, z_mean, z_logvar):
        std = torch.exp(0.5 * z_logvar)
        epsilon = torch.randn_like(std)
        return z_mean + std * epsilon

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_out(h))

    def forward(self, x):
        z_mean, z_logvar = self.encode(x)
        z = self.reparameterize(z_mean, z_logvar)
        reconstruction = self.decode(z)
        return reconstruction, z_mean, z_logvar

def vae_loss(reconstruction, x, z_mean, z_logvar):
    recon_loss = F.binary_cross_entropy(reconstruction, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(
        1 + z_logvar - z_mean.pow(2) - z_logvar.exp()
    )
    return recon_loss + kl_loss, recon_loss, kl_loss

model = VAE(original_dim, latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 15

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_recon, total_kl, n_samples = 0.0, 0.0, 0.0, 0

    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()

        reconstruction, z_mean, z_logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(
            reconstruction, x, z_mean, z_logvar
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        n_samples += x.size(0)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Loss: {total_loss / n_samples:.4f} | "
        f"Reconstruction: {total_recon / n_samples:.4f} | "
        f"KL: {total_kl / n_samples:.4f}"
    )

model.eval()
n = 15
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.tensor(
                [[xi, yi]], dtype=torch.float32
            ).to(device)
            x_decoded = model.decode(z_sample).cpu().numpy()
            digit = x_decoded[0].reshape(digit_size, digit_size)
            figure[
                i * digit_size:(i + 1) * digit_size,
                j * digit_size:(j + 1) * digit_size
            ] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="Greys_r")
plt.axis("off")
plt.title("Digits Generated by VAE")
plt.show()

all_z_mean = []
all_labels = []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        z_mean, _ = model.encode(x)
        all_z_mean.append(z_mean.cpu().numpy())
        all_labels.append(y.numpy())

all_z_mean = np.concatenate(all_z_mean, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    all_z_mean[:, 0],
    all_z_mean[:, 1],
    c=all_labels,
    cmap="tab10",
    s=2
)
plt.colorbar(scatter, label="Digit Label")
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("MNIST Digits in 2D Latent Space")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
MAZE = [
    "S.#.....",
    ".##.###.",
    "....#...",
    ".####.#.",
    "......#.",
    ".####.#G",
    "....#...",
    "###...#.",]

DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ARROWS = ["U", "D", "L", "R"]
ALPHA, GAMMA = 0.1, 0.95
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.01, 0.995
EPISODES, MAX_STEPS = 2000, 200
R_GOAL, R_STEP, R_WALL = 100.0, -1.0, -5.0

rng = np.random.default_rng(42)

class MazeEnv:
    def __init__(self, layout):
        self.grid = np.array([list(row) for row in layout])
        self.n_rows, self.n_cols = self.grid.shape
        self.start = tuple(map(int, np.argwhere(self.grid == "S")[0]))
        self.goal = tuple(map(int, np.argwhere(self.grid == "G")[0]))
        self.pos = self.start
    def reset(self):
        self.pos = self.start
        return self.pos
    def step(self, action):
        dr, dc = DELTAS[action]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        if not (0 <= r < self.n_rows and 0 <= c < self.n_cols) or self.grid[r, c] == "#":
            return self.pos, R_WALL, False
        self.pos = (r, c)
        done = self.pos == self.goal
        return self.pos, R_GOAL if done else R_STEP, done
def choose_action(Q, state, epsilon):
    if rng.random() < epsilon:
        return int(rng.integers(len(DELTAS)))
    q = Q[state]
    return int(rng.choice(np.flatnonzero(q == q.max())))

def train(env):
    Q = np.zeros((env.n_rows, env.n_cols, len(DELTAS)))
    epsilon, rewards, steps_per_ep = EPS_START, [], []

    for ep in range(EPISODES):
        state = env.reset()
        total, steps, done = 0.0, 0, False

        while not done and steps < MAX_STEPS:
            action = choose_action(Q, state, epsilon)
            next_state, reward, done = env.step(action)
            best_next = 0.0 if done else np.max(Q[next_state])
            Q[state][action] += ALPHA * (reward + GAMMA * best_next - Q[state][action])
            state, total, steps = next_state, total + reward, steps + 1

        epsilon = max(EPS_END, epsilon * EPS_DECAY)
        rewards.append(total)
        steps_per_ep.append(steps)

        if (ep + 1) % 200 == 0:
            print(f"Episode {ep + 1:5d} | avg reward (last 200): "
                  f"{np.mean(rewards[-200:]):7.1f} | avg steps: "
                  f"{np.mean(steps_per_ep[-200:]):6.1f} | epsilon: {epsilon:.3f}")

    return Q, rewards, steps_per_ep

def greedy_path(env, Q):
    state, path = env.reset(), []
    path.append(state)

    for _ in range(env.n_rows * env.n_cols):
        state, _, done = env.step(int(np.argmax(Q[state])))
        path.append(state)
        if done:
            return path, True

    return path, False

def moving_avg(x, w=50):
    return np.convolve(x, np.ones(w) / w, mode="valid")

def plot_results(env, Q, rewards, steps_per_ep, path):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    for ax, data, label, title, ylabel in zip(
        axes[:2],
        (rewards, steps_per_ep),
        ("Episode reward", "Steps"),
        ("Reward per episode", "Steps to finish episode"),
        ("Total reward", "Steps")
    ):
        ax.plot(data, alpha=0.3, label=label)
        ax.plot(range(49, len(data)), moving_avg(data), label="Moving avg (50)")
        ax.set(title=title, xlabel="Episode", ylabel=ylabel)
        ax.legend()

    ax = axes[2]
    img = np.zeros((env.n_rows, env.n_cols))
    img[env.grid == "#"] = 1
    ax.imshow(img, cmap="Greys", vmin=0, vmax=1.5)

    for r in range(env.n_rows):
        for c in range(env.n_cols):
            if env.grid[r, c] == "#":
                continue
            if (r, c) == env.start:
                ax.text(c, r, "S", ha="center", va="center", color="green", fontsize=14, fontweight="bold")
            elif (r, c) == env.goal:
                ax.text(c, r, "G", ha="center", va="center", color="red", fontsize=14, fontweight="bold")
            elif Q[r, c].max() != 0:
                ax.text(c, r, ARROWS[int(np.argmax(Q[r, c]))], ha="center", va="center", color="steelblue", fontsize=14)

    ys, xs = zip(*path)
    ax.plot(xs, ys, "o-", color="orange", linewidth=2, markersize=4, alpha=0.8, label="Greedy path")
    ax.set_xticks(range(env.n_cols))
    ax.set_yticks(range(env.n_rows))
    ax.set_title("Learned policy & optimal path")
    ax.legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    env = MazeEnv(MAZE)
    Q, rewards, steps_per_ep = train(env)
    path, reached = greedy_path(env, Q)

    print("\nGreedy path reached goal:", reached)
    print("Path length (steps):", len(path) - 1)
    print("Path:", path)

    plot_results(env, Q, rewards, steps_per_ep, path)